In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Invoke only the hash-pinned serving source. No mounted command list is executed.
assert not TRUE_SUBMISSION, 'startup smoke is not a competition submission'
source_slug = 'duck-qwen38-nvfp4-mtp-vllm-smoke-v1'
candidates = [Path('/kaggle/input/datasets/keithtyser') / source_slug, Path('/kaggle/input') / source_slug]
matching = [p for p in candidates if (p / 'serving_setup.py').is_file()]
assert len(matching) == 1, f'Expected one known source mount, got {matching}'
BUNDLE_DIR = matching[0]
assert __import__('hashlib').sha256((BUNDLE_DIR / 'serving_setup.py').read_bytes()).hexdigest() == '037c041c9bd9dcffa9084b32f47af9cf1bf35849d5eaf2e1a3422daac098b2e2', 'source drift'
SETUP_ENV_PATH = WORKING_DIR / 'startup-smoke-env.json'
SETUP_ENV_PATH.write_text('{}')
env = os.environ.copy()
env.update(TAAF_KAGGLE_BUNDLE_DIR=str(BUNDLE_DIR), TAAF_KAGGLE_WORKING_DIR=str(WORKING_DIR), TAAF_KAGGLE_SETUP_ENV=str(SETUP_ENV_PATH), TAAF_KAGGLE_FAST_START='0')
try:
    subprocess.run([sys.executable, '-c', "import hashlib,json\nfrom pathlib import Path\noriginal=Path(__import__('os').environ['TAAF_KAGGLE_BUNDLE_DIR'])/'serving_setup.py'\nsource=original.read_text()\nassert hashlib.sha256(source.encode()).hexdigest()=='037c041c9bd9dcffa9084b32f47af9cf1bf35849d5eaf2e1a3422daac098b2e2', 'source drift'\nfor before,after in {'SERVER_READY_TIMEOUT = 1_500': 'SERVER_READY_TIMEOUT = 2_400', 'SETUP_TOTAL_TIMEOUT = 1_800': 'SETUP_TOTAL_TIMEOUT = 2_700'}.items():\n    assert source.count(before)==1, 'ambiguous patch'\n    source=source.replace(before,after)\nPath('/kaggle/working/startup_patch_receipt.json').write_text(json.dumps({'original_sha256':'037c041c9bd9dcffa9084b32f47af9cf1bf35849d5eaf2e1a3422daac098b2e2','executed_sha256':hashlib.sha256(source.encode()).hexdigest(),'server_ready_timeout':2400,'setup_total_timeout':2700,'session_timeout_required':3600},sort_keys=True))\nscope = {'__name__': '__main__', '__file__': str(original)}\nsetup_completed = False\ntry:\n    exec(compile(source, str(original), 'exec'), scope)\n    setup_completed = True\nfinally:\n    cleanup = scope['stop_owned_server']() if 'stop_owned_server' in scope else {'stopped': False, 'error': 'cleanup unavailable'}\n    record = {'setup_completed': setup_completed, 'cleanup': cleanup, 'gameplay_executed': False}\n    if setup_completed:\n        record['preflight'] = json.loads(scope['PREFLIGHT_PATH'].read_text())\n        record['provenance'] = json.loads(scope['SETUP_PROVENANCE'].read_text())\n    record['passed'] = bool(setup_completed and cleanup.get('stopped') and cleanup.get('port_open') is False and cleanup.get('survivors') == [] and not record.get('preflight', {}).get('skipped', False) and {'text', 'tool', 'vision', 'c1_throughput', 'c4_throughput', 'native_mtp'} <= record.get('preflight', {}).keys())\n    scope.get('WORKING_DIR', original.parent).joinpath('startup_smoke_result.json').write_text(json.dumps(record, sort_keys=True, indent=2))\n    if setup_completed and not record['passed']:\n        raise RuntimeError('startup smoke did not pass preflight and cleanup')\n\n"], check=True, env=env, cwd=WORKING_DIR, timeout=3000)
except subprocess.TimeoutExpired:
    # Killing the setup child does not prove its separately-sessioned vLLM server stopped.
    # Use the same source's identity-bound cleanup, never a broad process kill.
    subprocess.run([sys.executable, '-c', "import hashlib,json,os\nfrom pathlib import Path\noriginal=Path(os.environ['TAAF_KAGGLE_BUNDLE_DIR'])/'serving_setup.py'\nsource=original.read_text()\nassert hashlib.sha256(source.encode()).hexdigest()=='037c041c9bd9dcffa9084b32f47af9cf1bf35849d5eaf2e1a3422daac098b2e2', 'cleanup source drift'\nscope={'__name__':'startup_cleanup_only','__file__':str(original)}\nexec(compile(source,str(original),'exec'),scope)\nresult=scope['stop_owned_server']()\nscope['WORKING_DIR'].joinpath('startup_timeout_cleanup.json').write_text(json.dumps(result,sort_keys=True))\n"], check=True, env=env, cwd=WORKING_DIR, timeout=150)
    raise
